# Cost & Consumption Governance

CalOptima RFP 26-038 | Topic — Cost Management & Spend Control

Source: `SNOWFLAKE.ACCOUNT_USAGE` (45-min latency) — requires ACCOUNTADMIN or SNOWFLAKE database grants.

**What this notebook covers:**
- Executive summary of credit consumption by service type
- Warehouse, user, and role-level spend breakdown
- Most expensive individual queries via Query Attribution
- Storage, serverless, and pipeline service costs
- Daily spend trends and hottest tables
- Live budget alerts, resource monitors, and spend-control workflow

In [ ]:
%%sql
USE ROLE      ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;
USE DATABASE  SNOWFLAKE;

## Section 1 — Executive Summary

Total credits consumed **this calendar month** across all Snowflake service types. Good opening view for an exec — shows where the money is going at the highest level.

In [ ]:
%%sql
SELECT
    SERVICE_TYPE,
    ROUND(SUM(CREDITS_USED), 2)                AS total_credits,
    ROUND(SUM(CREDITS_USED_COMPUTE), 2)        AS compute_credits,
    ROUND(SUM(CREDITS_USED_CLOUD_SERVICES), 2) AS cloud_svc_credits
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_HISTORY
WHERE START_TIME >= DATE_TRUNC('month', CURRENT_DATE())
GROUP BY SERVICE_TYPE
ORDER BY total_credits DESC;

## Section 2 — Credits by Warehouse (Last 30 Days)

Identifies which warehouses are driving the most spend. If a warehouse consumes disproportionate credits but runs only a handful of jobs, it is likely oversized.

In [ ]:
%%sql
SELECT
    WAREHOUSE_NAME,
    ROUND(SUM(CREDITS_USED), 2)                AS total_credits,
    ROUND(SUM(CREDITS_USED_COMPUTE), 2)        AS compute_credits,
    ROUND(SUM(CREDITS_USED_CLOUD_SERVICES), 2) AS cloud_svc_credits,
    COUNT(*)                                   AS billing_periods,
    MIN(START_TIME)::DATE                      AS first_active,
    MAX(START_TIME)::DATE                      AS last_active
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY WAREHOUSE_NAME
ORDER BY total_credits DESC;

## Section 3 — Most Expensive Queries (Query Attribution)

`QUERY_ATTRIBUTION_HISTORY` attributes compute credits directly to individual queries — the most precise view for pinpointing costly SQL. Joined to `QUERY_HISTORY` to surface actual query text, user, warehouse, and bytes scanned.

In [ ]:
%%sql
SELECT
    QA.QUERY_ID,
    QH.QUERY_TEXT,
    QH.USER_NAME,
    QH.ROLE_NAME,
    QH.WAREHOUSE_NAME,
    QH.DATABASE_NAME,
    ROUND(QA.CREDITS_ATTRIBUTED_COMPUTE, 6) AS credits_attributed,
    QH.EXECUTION_TIME / 1000                AS execution_secs,
    QH.BYTES_SCANNED / 1e9                  AS gb_scanned,
    QH.QUERY_START_TIME::DATE               AS query_date
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY QA
JOIN SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY             QH
    ON QA.QUERY_ID = QH.QUERY_ID
WHERE QA.QUERY_START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
  AND QA.CREDITS_ATTRIBUTED_COMPUTE > 0
ORDER BY credits_attributed DESC
LIMIT 25;

## Section 4 — Most Expensive Users (Last 30 Days)

Ranks users by total compute credits consumed. Surfaces power users and runaway workloads. The `max_single_query` column flags anyone who fired a single large scan.

In [ ]:
%%sql
SELECT
    QH.USER_NAME,
    COUNT(DISTINCT QH.QUERY_ID)                  AS query_count,
    ROUND(SUM(QA.CREDITS_ATTRIBUTED_COMPUTE), 4) AS total_credits,
    ROUND(AVG(QA.CREDITS_ATTRIBUTED_COMPUTE), 6) AS avg_credits_per_query,
    ROUND(MAX(QA.CREDITS_ATTRIBUTED_COMPUTE), 6) AS max_single_query,
    ROUND(SUM(QH.BYTES_SCANNED) / 1e12, 3)      AS tb_scanned
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY QA
JOIN SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY             QH
    ON QA.QUERY_ID = QH.QUERY_ID
WHERE QA.QUERY_START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY QH.USER_NAME
ORDER BY total_credits DESC
LIMIT 20;

## Section 5 — Most Expensive Roles (Last 30 Days)

Role-level spend is the foundation for **chargeback and cost allocation** — each role typically maps to a team or business function. Lets finance allocate Snowflake costs back to the departments that drove them.

In [ ]:
%%sql
SELECT
    QH.ROLE_NAME,
    COUNT(DISTINCT QH.USER_NAME)                 AS distinct_users,
    COUNT(DISTINCT QH.QUERY_ID)                  AS query_count,
    ROUND(SUM(QA.CREDITS_ATTRIBUTED_COMPUTE), 4) AS total_credits,
    ROUND(AVG(QA.CREDITS_ATTRIBUTED_COMPUTE), 6) AS avg_credits_per_query,
    ROUND(SUM(QH.BYTES_SCANNED) / 1e12, 3)      AS tb_scanned
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY QA
JOIN SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY             QH
    ON QA.QUERY_ID = QH.QUERY_ID
WHERE QA.QUERY_START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY QH.ROLE_NAME
ORDER BY total_credits DESC
LIMIT 20;

## Section 6 — Storage Costs by Database (Last 30 Days)

Storage is billed on average daily bytes. Breaks down into **active storage** (current data) vs **fail-safe** (7-day disaster recovery copy). Large fail-safe numbers mean high-churn tables — candidates for shorter retention settings.

In [ ]:
%%sql
SELECT
    DATABASE_NAME,
    ROUND(AVG(AVERAGE_DATABASE_BYTES) / 1e9, 2)                          AS avg_active_gb,
    ROUND(AVG(AVERAGE_FAILSAFE_BYTES) / 1e9, 2)                          AS avg_failsafe_gb,
    ROUND(AVG(AVERAGE_DATABASE_BYTES + AVERAGE_FAILSAFE_BYTES) / 1e9, 2) AS avg_total_gb,
    MAX(USAGE_DATE)                                                       AS last_recorded
FROM SNOWFLAKE.ACCOUNT_USAGE.DATABASE_STORAGE_USAGE_HISTORY
WHERE USAGE_DATE >= DATEADD('day', -30, CURRENT_DATE())
GROUP BY DATABASE_NAME
ORDER BY avg_total_gb DESC;

## Section 7 — Serverless & Pipeline Service Costs (Last 30 Days)

Credits consumed by **serverless features** — these run outside warehouses and are easy to overlook. Covers: serverless tasks, Snowpipe, auto-clustering, search optimization, and replication.

In [ ]:
%%sql
SELECT 'SERVERLESS_TASK'     AS service, ROUND(SUM(CREDITS_USED), 4) AS credits
FROM SNOWFLAKE.ACCOUNT_USAGE.SERVERLESS_TASK_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
UNION ALL
SELECT 'SNOWPIPE',            ROUND(SUM(CREDITS_USED), 4)
FROM SNOWFLAKE.ACCOUNT_USAGE.PIPE_USAGE_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
UNION ALL
SELECT 'AUTO_CLUSTERING',     ROUND(SUM(CREDITS_USED), 4)
FROM SNOWFLAKE.ACCOUNT_USAGE.AUTOMATIC_CLUSTERING_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
UNION ALL
SELECT 'SEARCH_OPTIMIZATION', ROUND(SUM(CREDITS_USED), 4)
FROM SNOWFLAKE.ACCOUNT_USAGE.SEARCH_OPTIMIZATION_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
UNION ALL
SELECT 'REPLICATION',         ROUND(SUM(CREDITS_USED), 4)
FROM SNOWFLAKE.ACCOUNT_USAGE.REPLICATION_GROUP_USAGE_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
ORDER BY credits DESC;

## Section 8 — Daily Credit Trend (Last 30 Days)

Rolling daily spend — ideal for spotting anomalies (a spike on a weekend), growth trends, or the impact of a new workload being onboarded.

In [ ]:
%%sql
SELECT
    DATE_TRUNC('day', START_TIME)::DATE        AS usage_date,
    ROUND(SUM(CREDITS_USED_COMPUTE), 2)        AS compute_credits,
    ROUND(SUM(CREDITS_USED_CLOUD_SERVICES), 2) AS cloud_svc_credits,
    ROUND(SUM(CREDITS_USED), 2)                AS total_credits
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY usage_date
ORDER BY usage_date;

## Section 9 — Hottest Tables by Query Frequency (Last 7 Days)

Surfaces the most-queried tables — the best candidates for **clustering keys**, **search optimization**, or **materialized views**. High GB scanned + high query count = high optimization ROI.

In [ ]:
%%sql
SELECT
    BASE_OBJECTS_ACCESSED[0]:objectName::VARCHAR AS table_name,
    COUNT(*)                                      AS query_count,
    COUNT(DISTINCT USER_NAME)                     AS distinct_users,
    ROUND(SUM(BYTES_SCANNED) / 1e9, 2)           AS total_gb_scanned,
    ROUND(AVG(EXECUTION_TIME) / 1000, 1)         AS avg_exec_secs
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE QUERY_START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
  AND EXECUTION_STATUS = 'SUCCESS'
  AND ARRAY_SIZE(BASE_OBJECTS_ACCESSED) > 0
GROUP BY table_name
ORDER BY query_count DESC
LIMIT 20;

## Section 10 — Budget Setup (Snowflake Budgets)

Snowflake Budgets track credit consumption against a defined monthly limit and send email notifications at configurable thresholds. This is the **proactive** spend-control layer — alerts fire before limits are hit.

> Adjust `CREDIT_QUOTA` to match your contract. Notification integration must exist before running.

In [ ]:
%%sql
-- Step 1: Email notification integration
CREATE NOTIFICATION INTEGRATION IF NOT EXISTS BUDGET_EMAIL_INTEGRATION
    TYPE = EMAIL
    ENABLED = TRUE;

-- Step 2: Account-level budget with 50% and 90% email alerts
CREATE BUDGET IF NOT EXISTS SNOWFLAKE.LOCAL.CALOPTIMA_ACCOUNT_BUDGET
    CREDIT_QUOTA = 1000
    NOTIFY (
        THRESHOLD = 50 PERCENT NOTIFY (INTEGRATION = BUDGET_EMAIL_INTEGRATION),
        THRESHOLD = 90 PERCENT NOTIFY (INTEGRATION = BUDGET_EMAIL_INTEGRATION)
    );

-- Step 3: Check current budget status
SELECT *
FROM TABLE(SNOWFLAKE.LOCAL.GET_BUDGET_ALLOWANCE(
    BUDGET_NAME => 'SNOWFLAKE.LOCAL.CALOPTIMA_ACCOUNT_BUDGET'
));

## Section 11 — Resource Monitors (Warehouse-Level Spend Control)

Resource monitors are the **enforcement** layer — they actively suspend warehouses when credit thresholds are breached.

| Threshold | Action |
|---|---|
| 80% | NOTIFY — alert only, keep running |
| 100% | SUSPEND — finish in-flight queries, then stop |
| 110% | SUSPEND_IMMEDIATE — kill all queries immediately |

> Adjust `CREDIT_QUOTA` and attach the monitor to any warehouse that needs a hard cap.

In [ ]:
%%sql
CREATE OR REPLACE RESOURCE MONITOR CALOPTIMA_MONTHLY_MONITOR
    WITH
        CREDIT_QUOTA    = 500
        FREQUENCY       = MONTHLY
        START_TIMESTAMP = IMMEDIATELY
    TRIGGERS
        ON 80  PERCENT DO NOTIFY
        ON 100 PERCENT DO SUSPEND
        ON 110 PERCENT DO SUSPEND_IMMEDIATE;

ALTER WAREHOUSE COMPUTE_WH
    SET RESOURCE_MONITOR = CALOPTIMA_MONTHLY_MONITOR;

## Section 12 — Custom Daily Spend Alert

A Snowflake `ALERT` that runs at 8am every morning, checks yesterday's total credits from `METERING_DAILY_HISTORY`, and emails if the threshold is exceeded. This is the **reactive** layer — catches overspend within hours.

> Adjust the credit threshold (currently `50`) and recipient email to match your environment.

In [ ]:
%%sql
CREATE OR REPLACE ALERT SNOWFLAKE.LOCAL.DAILY_SPEND_ALERT
    WAREHOUSE = COMPUTE_WH
    SCHEDULE  = 'USING CRON 0 8 * * * America/Los_Angeles'
    IF (
        EXISTS (
            SELECT 1
            FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
            WHERE USAGE_DATE = CURRENT_DATE() - 1
            HAVING SUM(CREDITS_USED) > 50
        )
    )
    THEN
        CALL SYSTEM$SEND_EMAIL(
            'BUDGET_EMAIL_INTEGRATION',
            'caloptima-snowflake-alerts@caloptima.org',
            'Snowflake Daily Spend Alert',
            'Yesterday''s credit consumption exceeded the 50-credit daily threshold. '
            || 'Log in to Snowsight -> Admin -> Cost Management to review.'
        );

ALTER ALERT SNOWFLAKE.LOCAL.DAILY_SPEND_ALERT RESUME;

## Section 13 — Verify: Active Monitors & Alerts

Confirm resource monitors and the daily spend alert are active, and check current consumption vs quota.

In [ ]:
%%sql
SHOW RESOURCE MONITORS;

SHOW ALERTS LIKE 'DAILY_SPEND_ALERT%' IN SCHEMA SNOWFLAKE.LOCAL;

SELECT
    NAME,
    CREDIT_QUOTA,
    CREDITS_USED,
    ROUND(CREDITS_USED / CREDIT_QUOTA * 100, 1) AS pct_used,
    FREQUENCY,
    START_TIME,
    END_TIME
FROM SNOWFLAKE.ACCOUNT_USAGE.RESOURCE_MONITORS
ORDER BY pct_used DESC NULLS LAST;